# CS781 Phase 1 — Decoder-only Transformer

This notebook reuses the **verified Phase-4 data foundation** already prepared in the project:
- 32K legal BPE tokenizer
- 4096-token context
- train/validation/test manifests
- byte-offset source index
- deduplicated legal corpus
- recovery/training configuration artifacts

It is designed to be **run once and resume automatically**.

> **Important:** the architecture cell is intentionally left for the student's course-derived decoder Transformer implementation. The notebook does not invent or replace that architecture.


In [ ]:
# ============================================================
# 1. ROBUST PROJECT / ARTIFACT DISCOVERY + TRAINING CONFIG
# ============================================================

from pathlib import Path
import os, json, random, math, zipfile, time, gc
import numpy as np
import torch
from tokenizers import Tokenizer
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

INPUT = Path("/kaggle/input")
RUN_DIR = Path("/kaggle/working/cs781_phase1_decoder_transformer")
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CONTEXT_LENGTH = 4096
VOCAB_SIZE = 32_000

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_WORKERS = 0

VAL_EVERY = 250
SAVE_EVERY = 250
MAX_VAL_BATCHES = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (
    torch.bfloat16
    if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def all_named_files(name):
    return sorted(INPUT.rglob(name))

def choose_phase4_root():
    candidates = []
    for p in all_named_files("train_manifest.jsonl"):
        root = p.parent.parent
        if (root / "tokenizer" / "legal_bpe_32k.json").exists() and (root / "manifests" / "validation_manifest.jsonl").exists():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "Could not locate the Phase-4 training-prep dataset. "
            "Attach cs781-phase4-training-prep to this Kaggle notebook."
        )
    # Prefer the explicit final-recovery/training-prep layout if present.
    candidates = sorted(candidates, key=lambda x: str(x))
    for c in candidates:
        if "cs781-phase4-training-prep" in str(c):
            return c
    return candidates[0]

CHECKPOINT_ROOT = choose_phase4_root()

TOKENIZER_PATH = CHECKPOINT_ROOT / "tokenizer" / "legal_bpe_32k.json"
TRAIN_MANIFEST = CHECKPOINT_ROOT / "manifests" / "train_manifest.jsonl"
VAL_MANIFEST = CHECKPOINT_ROOT / "manifests" / "validation_manifest.jsonl"
TEST_MANIFEST = CHECKPOINT_ROOT / "manifests" / "test_manifest.jsonl"

# The earlier Phase-4 work built a byte-offset source index.
index_candidates = (
    list(CHECKPOINT_ROOT.rglob("source_index.jsonl"))
    + list(CHECKPOINT_ROOT.rglob("*source_index*.jsonl"))
)
if not index_candidates:
    raise FileNotFoundError("No source index found in the attached Phase-4 artifacts.")
INDEX_PATH = sorted(set(index_candidates), key=lambda p: p.stat().st_size, reverse=True)[0]

dedup_candidates = [p for p in CHECKPOINT_ROOT.rglob("dedup") if p.is_dir()]
if not dedup_candidates:
    raise FileNotFoundError("No dedup corpus directory found.")
DEDUP_ROOT = dedup_candidates[0]

# ------------------------------------------------------------
# Auto-read the previously documented training/recovery config.
# Do NOT invent LR/WD/steps/clip values here.
# ------------------------------------------------------------
def find_config(name):
    hits = list(INPUT.rglob(name))
    # Prefer the final-recovery copy if multiple datasets are attached.
    hits = sorted(hits, key=lambda p: ("cs781-phase4-final-recovery" not in str(p), str(p)))
    return hits[0] if hits else None

TRAINING_STATE_PATH = find_config("training_state.json")
RECOVERY_CONFIG_PATH = find_config("phase4_recovery_config.json")
MODEL_CONFIG_PATH = find_config("model_architecture.json")
EXAMPLE_CONFIG_PATH = find_config("training_example_config.json")

def load_json(path):
    if path is None:
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

training_state = load_json(TRAINING_STATE_PATH)
recovery_config = load_json(RECOVERY_CONFIG_PATH)
model_config = load_json(MODEL_CONFIG_PATH)
example_config = load_json(EXAMPLE_CONFIG_PATH)

def first_value(*dicts, keys):
    for d in dicts:
        for k in keys:
            if k in d and d[k] is not None:
                return d[k]
    return None

LR = first_value(training_state, recovery_config, example_config,
                 keys=("learning_rate", "lr"))
WEIGHT_DECAY = first_value(training_state, recovery_config, example_config,
                           keys=("weight_decay",))
MAX_STEPS = first_value(training_state, recovery_config, example_config,
                        keys=("max_steps", "num_steps", "training_steps"))
MAX_GRAD_NORM = first_value(training_state, recovery_config, example_config,
                            keys=("max_grad_norm", "grad_clip", "gradient_clip_norm"))

# Normalize numeric types if the JSON stores them as strings.
def as_float(x):
    return None if x is None else float(x)

def as_int(x):
    return None if x is None else int(x)

LR = as_float(LR)
WEIGHT_DECAY = as_float(WEIGHT_DECAY)
MAX_STEPS = as_int(MAX_STEPS)
MAX_GRAD_NORM = as_float(MAX_GRAD_NORM)

print("=" * 80)
print("CS781 PHASE 1 — DECODER TRANSFORMER CONFIGURATION")
print("=" * 80)
print("Device             :", DEVICE)
print("Checkpoint root    :", CHECKPOINT_ROOT)
print("Tokenizer          :", TOKENIZER_PATH)
print("Train manifest     :", TRAIN_MANIFEST)
print("Validation manifest:", VAL_MANIFEST)
print("Test manifest      :", TEST_MANIFEST)
print("Source index       :", INDEX_PATH)
print("Dedup root         :", DEDUP_ROOT)
print()
print("Context length     :", CONTEXT_LENGTH)
print("Vocab size         :", VOCAB_SIZE)
print("Batch size         :", BATCH_SIZE)
print("Grad accumulation  :", GRAD_ACCUM_STEPS)
print("Workers            :", NUM_WORKERS)
print()
print("LR                 :", LR)
print("Weight decay       :", WEIGHT_DECAY)
print("Max steps          :", MAX_STEPS)
print("Max grad norm      :", MAX_GRAD_NORM)

if None in (LR, WEIGHT_DECAY, MAX_STEPS, MAX_GRAD_NORM):
    raise RuntimeError(
        "The attached recovery artifacts do not contain all four training values. "
        "Run the Phase-4 configuration/recovery notebook that produced training_state.json, "
        "or explicitly set these four values from your documented course-approved design."
    )

assert TOKENIZER_PATH.exists()
assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()
assert TEST_MANIFEST.exists()
assert INDEX_PATH.exists()
assert DEDUP_ROOT.exists()

print("\n✓ Required Phase-4 artifacts found")
print("✓ Training settings loaded from existing project artifacts")


In [ ]:
# ============================================================
# 2. TOKENIZER + BYTE-OFFSET SOURCE READER
# ============================================================

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
assert tokenizer.get_vocab_size() == VOCAB_SIZE

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")
assert None not in (PAD_ID, BOS_ID, EOS_ID)

# Load only the compact byte-offset index.
source_lookup = {}
with open(INDEX_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            source_lookup[r["text_hash"]] = r

print("Tokenizer vocab :", tokenizer.get_vocab_size())
print("PAD/BOS/EOS     :", PAD_ID, BOS_ID, EOS_ID)
print("Index entries   :", f"{len(source_lookup):,}")

def read_record(rec):
    r = source_lookup[rec["text_hash"]]
    rel = str(r["source_file"])
    if rel.endswith(".gz"):
        rel = rel[:-3]
    p = DEDUP_ROOT / rel
    if not p.exists():
        # Robust fallback for source paths containing a dataset-relative prefix.
        matches = list(DEDUP_ROOT.rglob(Path(rel).name))
        if len(matches) != 1:
            raise FileNotFoundError(f"Could not uniquely resolve {rel}")
        p = matches[0]

    with open(p, "rb") as f:
        f.seek(int(r["byte_offset"]))
        raw = f.read(int(r["byte_length"]))

    return json.loads(raw.rstrip(b"\r\n").decode("utf-8"))

def text_of(x):
    if isinstance(x, str):
        return x
    for k in ("text", "content", "document", "body"):
        if isinstance(x, dict) and isinstance(x.get(k), str):
            return x[k]
    raise KeyError(f"No text field found. Keys={list(x) if isinstance(x, dict) else type(x)}")

print("✓ Tokenizer and source reader ready")


In [ ]:
# ============================================================
# 3. MEMORY-SAFE WORKER-AWARE STREAMING DATASET
# Reuses the same manifest/index idea already verified in Phase 4.
# It does NOT materialize the 3M-document manifest as Python dicts.
# ============================================================

class WorkerSafeLegalStream(IterableDataset):
    def __init__(self, manifest_path, split):
        super().__init__()
        self.manifest_path = str(manifest_path)
        self.split = split

        # Keep byte offsets into the manifest rather than 3M JSON objects.
        self.offsets = []
        with open(self.manifest_path, "rb") as f:
            while True:
                pos = f.tell()
                line = f.readline()
                if not line:
                    break
                if line.strip():
                    self.offsets.append(pos)

    def _read_manifest_record(self, f, offset):
        f.seek(offset)
        return json.loads(f.readline())

    def __iter__(self):
        wi = get_worker_info()
        worker_id = 0 if wi is None else wi.id
        num_workers = 1 if wi is None else wi.num_workers

        with open(self.manifest_path, "rb") as mf:
            for j in range(worker_id, len(self.offsets), num_workers):
                rec = self._read_manifest_record(mf, self.offsets[j])
                ids = [BOS_ID] + tokenizer.encode(text_of(read_record(rec))).ids + [EOS_ID]

                # Document-aware fixed context chunks, same 4096 context.
                for start in range(0, len(ids), CONTEXT_LENGTH):
                    x = ids[start:start + CONTEXT_LENGTH]
                    if len(x) < CONTEXT_LENGTH:
                        x = x + [PAD_ID] * (CONTEXT_LENGTH - len(x))

                    x = torch.tensor(x, dtype=torch.long)
                    y = x.clone()
                    y[y == PAD_ID] = -100

                    yield {
                        "input_ids": x,
                        "labels": y,
                        "attention_mask": (x != PAD_ID).long(),
                    }

train_ds = WorkerSafeLegalStream(TRAIN_MANIFEST, "train")
val_ds = WorkerSafeLegalStream(VAL_MANIFEST, "validation")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train manifest offsets:", f"{len(train_ds.offsets):,}")
print("Val manifest offsets  :", f"{len(val_ds.offsets):,}")
print("✓ Lazy manifest streaming enabled")


## 4. Decoder Transformer architecture — student implementation

This is the **only substantive implementation cell intentionally not auto-filled**.

Use your existing course-derived decoder-only Transformer implementation here. It must be randomly initialized and expose:

```text
model(input_ids=..., attention_mask=..., labels=...)
    -> object with .loss and .logits
```

Expected reference configuration from the verified project artifacts is approximately **112.8M parameters**, with 32K vocabulary and 4096 context.

Do not load the Phase-4 checkpoint as weights for this Phase-1 model. The checkpoint is useful as a project/recovery artifact, but the Phase-1 model must be initialized from scratch.

After your implementation, the next cell verifies the parameter count and forward-pass contract.


In [ ]:
# ============================================================
# 5. MODEL CONTRACT / PARAMETER VERIFICATION
# ============================================================

# Replace ONLY the following line with your own course-derived
# decoder Transformer constructor.
#
# Example shape (not an architecture implementation):
# model = YourDecoderTransformer(...)
#
# The project artifacts indicate:
#   vocab_size=32000
#   context_length=4096
#   ~112.78M parameters
#
# model = YOUR_DECODER_TRANSFORMER_HERE

if "model" not in globals():
    raise RuntimeError(
        "Define `model` using your course-derived decoder-only Transformer "
        "implementation before running this cell."
    )

param_count = sum(p.numel() for p in model.parameters())
print(f"Parameter count: {param_count:,}")

assert 50_000_000 <= param_count <= 300_000_000, (
    f"Phase-1 model must be 50M–300M parameters; got {param_count:,}"
)

# Forward smoke test.
sample = next(iter(train_loader))
sample = {k: v.to(DEVICE) for k, v in sample.items()}

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=DEVICE.type == "cuda"
    ):
        out = model(
            input_ids=sample["input_ids"],
            attention_mask=sample["attention_mask"],
            labels=sample["labels"],
        )

assert hasattr(out, "loss")
assert hasattr(out, "logits")
assert out.logits.shape[:2] == sample["input_ids"].shape
assert out.logits.shape[-1] == VOCAB_SIZE
assert torch.isfinite(out.loss).item()

ARCH_NAME = "decoder_transformer"
print("✓ Parameter range valid")
print("✓ Forward contract valid")
print("✓ Logits shape:", tuple(out.logits.shape))
print("✓ Initial loss:", float(out.loss))


In [ ]:
# ============================================================
# 6. CHECKPOINT-SAFE TRAINING LOOP
# ============================================================

LATEST = RUN_DIR / "latest.pt"
BEST = RUN_DIR / "best.pt"
FINAL = RUN_DIR / "final.pt"

def rng_state():
    return {
        "py": random.getstate(),
        "np": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def restore_rng(s):
    random.setstate(s["py"])
    np.random.set_state(s["np"])
    torch.set_rng_state(s["torch"])
    if torch.cuda.is_available() and s["cuda"] is not None:
        torch.cuda.set_rng_state_all(s["cuda"])

model = model.to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.cuda.amp.GradScaler(
    enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16)
)

def save_checkpoint(path, step, seen_microbatches, best_val):
    tmp = str(path) + ".tmp"
    payload = {
        "architecture": ARCH_NAME,
        "step": int(step),
        "seen_microbatches": int(seen_microbatches),
        "best_val": float(best_val),
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "rng": rng_state(),
        "config": {
            "context_length": CONTEXT_LENGTH,
            "vocab_size": VOCAB_SIZE,
            "batch_size": BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
            "max_steps": MAX_STEPS,
            "max_grad_norm": MAX_GRAD_NORM,
            "seed": SEED,
        },
    }
    torch.save(payload, tmp)
    os.replace(tmp, path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if ckpt.get("architecture") != ARCH_NAME:
        raise RuntimeError("Checkpoint architecture does not match this notebook.")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    if ckpt.get("rng") is not None:
        restore_rng(ckpt["rng"])
    return (
        int(ckpt.get("step", 0)),
        int(ckpt.get("seen_microbatches", 0)),
        float(ckpt.get("best_val", float("inf"))),
    )

step = 0
seen_microbatches = 0
best_val = float("inf")

if LATEST.exists():
    step, seen_microbatches, best_val = load_checkpoint(LATEST)
    print(f"✓ Resuming from step={step}, microbatches={seen_microbatches:,}")
else:
    print("Starting from scratch.")

# Replaying the already-consumed prefix makes resume deterministic.
# This is slower at the beginning of a resumed run, but requires no
# manual dataset-state bookkeeping.
train_iter = iter(train_loader)
for _ in range(seen_microbatches):
    try:
        next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        next(train_iter)

def evaluate():
    model.eval()
    values = []
    vi = iter(val_loader)

    with torch.no_grad():
        for _ in range(MAX_VAL_BATCHES):
            try:
                batch = next(vi)
            except StopIteration:
                break

            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            with torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE,
                enabled=DEVICE.type == "cuda"
            ):
                result = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                )

            values.append(float(result.loss.detach().cpu()))

    model.train()
    return sum(values) / len(values) if values else float("inf")

model.train()
optimizer.zero_grad(set_to_none=True)

while step < MAX_STEPS:
    for _ in range(GRAD_ACCUM_STEPS):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=DEVICE.type == "cuda"
        ):
            loss = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            ).loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        seen_microbatches += 1

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM,
    )
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    step += 1

    if step % 10 == 0:
        print(
            f"step={step:,} "
            f"loss={float(loss.detach().cpu()) * GRAD_ACCUM_STEPS:.5f} "
            f"grad_norm={float(grad_norm):.4f}"
        )

    if step % VAL_EVERY == 0:
        val_loss = evaluate()
        print(f"validation_loss={val_loss:.5f}")

        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(BEST, step, seen_microbatches, best_val)
            print("✓ saved best.pt")

    if step % SAVE_EVERY == 0:
        save_checkpoint(LATEST, step, seen_microbatches, best_val)
        print("✓ saved latest.pt")

save_checkpoint(FINAL, step, seen_microbatches, best_val)
save_checkpoint(LATEST, step, seen_microbatches, best_val)

print("=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print("Final step :", step)
print("Best val   :", best_val)
print("Final      :", FINAL)


In [ ]:
# ============================================================
# 7. OFFICIAL TEST GENERATION + prediction.json + submission.zip
# ============================================================

model.eval()

def generate(prompt_ids, max_new_tokens=384):
    ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            x = ids[:, -CONTEXT_LENGTH:]
            mask = (x != PAD_ID).long()

            with torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE,
                enabled=DEVICE.type == "cuda"
            ):
                out = model(
                    input_ids=x,
                    attention_mask=mask
                )

            nxt = torch.argmax(
                out.logits[:, -1, :],
                dim=-1,
                keepdim=True
            )
            ids = torch.cat([ids, nxt], dim=1)

            if int(nxt.item()) == EOS_ID:
                break

    return ids[0].tolist()

preds = {}

with open(TEST_MANIFEST, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if not line.strip():
            continue

        rec = json.loads(line)
        prompt = text_of(read_record(rec))
        prompt_ids = tokenizer.encode(prompt).ids

        generated = generate(
            [BOS_ID] + prompt_ids,
            max_new_tokens=384,
        )

        continuation_ids = generated[len(prompt_ids) + 1:]
        generated_text = tokenizer.decode(
            continuation_ids,
            skip_special_tokens=True,
        )

        key = str(
            rec.get(
                "id",
                rec.get(
                    "document_id",
                    rec.get("text_hash")
                )
            )
        )

        preds[key] = {
            "text": " ".join(generated_text.split()[:128])
        }

        if line_no % 50 == 0:
            print("Generated:", line_no)

PRED = RUN_DIR / "prediction.json"
PRED.write_text(
    json.dumps(preds, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

ZIP = RUN_DIR / "submission.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(PRED, arcname="prediction.json")

print("prediction.json:", PRED)
print("submission.zip :", ZIP)
print("Predictions     :", len(preds))


## 8. What was fixed

Compared with the original 01 notebook:

1. **The LR/weight-decay/max-steps/max-grad-norm error is removed** by reading the existing recovery/training JSON artifacts automatically.
2. The notebook no longer eagerly converts the entire ~3M-record manifest into Python dictionaries.
3. Worker partitioning is deterministic and restartable.
4. Checkpoints contain model, optimizer, AMP scaler, RNG state, and training configuration.
5. `latest.pt` is overwritten atomically, so an interrupted write does not destroy the previous checkpoint.
6. Resume happens automatically when `latest.pt` exists.
7. The final cell creates `prediction.json` and `submission.zip`.

### Still required from you

Only the **decoder Transformer architecture implementation** itself remains student-owned. Put your course-derived implementation in Cell 5, instantiate `model`, and run the notebook top-to-bottom.

I have deliberately not invented the architecture implementation or silently chosen new hyperparameters for the course project.
